# Grid Creation

This code:

- Loads a 0.5° zonal grid (loop on a folder)

- Generates and saves the resolution local grids associated with each 0.5° tile

- Merge the cells together to create a resolution zonal grid

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
import os
import re

In [ ]:
# connection to the drive where the rasters are kept and where the results will be saved
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Here the parameters for the code to run smoothly are defined

# === PARAMETERS ===
# path to the folder with all grids at 0.05
GRILLE_05_PATH = "/content/drive/Zones_05"
# resolution at which you want to generate your new grid
RESOLUTION_FINE = 0.25
# path to the folder where your data will be saved
OUTPUT_FOLDER = "/content/drive/Zone_025"

# if the output folder does not exist, it is created
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
# === Create a grid at resolution required for an entire zone ===
def create_grid_resolution_for_zone(gdf_zone, zone_id):
    # create the empty list of cells
    all_cells = []

    # loop on the rows of the grid
    for idx, row in gdf_zone.iterrows():
        # get the bounds
        minx, miny, maxx, maxy = row.geometry.bounds

        if minx == maxx or miny == maxy:
            continue  # skip the geometry which area=0

        # create the step at the resolution required
        lon_steps = np.arange(minx, maxx, RESOLUTION_FINE)
        lat_steps = np.arange(miny, maxy, RESOLUTION_FINE)

        if len(lon_steps) == 0 or len(lat_steps) == 0:
            continue  # skip the geometry smaller than the resolution required

        # create an empty list for the cells of the row
        cells = []
        for lon in lon_steps:
            for lat in lat_steps:
                # create each cell and add them to the list
                cell = box(lon, lat, lon + RESOLUTION_FINE, lat + RESOLUTION_FINE)
                cells.append(cell)

        # if the list is empty, skip to the next row
        if not cells:
            continue

        # if the list is not empty, create the dataframe at the right projection
        gdf_cells = gpd.GeoDataFrame(geometry=cells, crs="EPSG:4326")
        gdf_cells["zone"] = zone_id
        gdf_cells["tile_id"] = idx
        # add the row of cells to the final list
        all_cells.append(gdf_cells)

    # merge the entire zone and create a geopackage to save it
    if all_cells:
        merged_gdf = gpd.GeoDataFrame(pd.concat(all_cells, ignore_index=True), crs="EPSG:4326")
        filename = f"grid_01_zone{zone_id}.gpkg"
        merged_gdf.to_file(os.path.join(OUTPUT_FOLDER, filename), driver="GPKG")
        print(f"✅ Zone {zone_id} finished and saved : {len(merged_gdf)} cells.")
    else:
        print(f"⚠️  Zone {zone_id} error : no cells created.")

# === Loop on each zone to create the resolution zonal grid ===
for i, filename in enumerate(os.listdir(GRILLE_05_PATH)):
    if filename.endswith(".shp"):

        # Extract the zone's number from the name of the file
        match = re.search(r"grilles_zone_(\d+)", filename) # rename the search depending on the filename format
        if match:
            zone_id = int(match.group(1))
        else:
            print(f"❌ Impossible to find the zone's number in this file : {filename}")
            continue

        filepath = os.path.join(GRILLE_05_PATH, filename)
        gdf_zone = gpd.read_file(filepath).reset_index(drop=True)
        print(f"\n⌛ Process on zone {zone_id} ({len(gdf_zone)} tiles) from {filename}...")

        create_grid_resolution_for_zone(gdf_zone, zone_id)

print("\n✅ All grids were processed.")